# NimbusPay ticket triage: LoRA fine-tune of Qwen2.5-3B-Instruct

Pipeline: setup, data audit and cleaning, sequence length analysis, baselines, training, controlled experiments, error analysis, final model and test predictions.

Runs on a free Colab T4. Every run saves its adapter, logs and predictions to Google Drive under `MyDrive/nimbus_triage/runs/`.

## 0. Setup

Pinned library versions, seeds, paths and shared helpers. On Colab the data is read from `MyDrive/nimbus_triage/candidate_pack/`; locally it is read from `./candidate_pack/`.

In [ ]:
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

IN_COLAB = "google.colab" in sys.modules

PINS = {
    "transformers": "5.22.1",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
}
EXPECTED_TORCH = "2.14.1"


def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


if IN_COLAB:
    missing = [f"{p}=={v}" for p, v in PINS.items() if installed(p) != v]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if installed("torchao"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

print("colab:", IN_COLAB)
print(f"torch=={installed('torch')}", "" if (installed("torch") or "").startswith(EXPECTED_TORCH) else f"(expected {EXPECTED_TORCH})")
for pkg, want in PINS.items():
    have = installed(pkg)
    print(f"{pkg}=={have}", "" if have == want else f"(expected {want})")
print("torchao:", installed("torchao") or "not installed (Colab's preinstalled 0.10 breaks peft's LoRA layer dispatch)")

In [ ]:
import csv
import json
import random
import time
import importlib.util
from datetime import datetime
from pathlib import Path

import numpy as np

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/nimbus_triage")
else:
    ROOT = Path.cwd()

PACK = ROOT / "candidate_pack"
DATA = PACK / "data"
RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SEED = 42
MAX_NEW_TOKENS = 200

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
CATEGORIES = ["payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"]
PRIORITIES = ["P1", "P2", "P3"]
CHANNELS = ["upi", "card", "netbanking", "wallet"]
LANGUAGES = ["en", "hi-en"]
SOURCES = ["app-chat", "email", "twitter", "ivr-transcript"]
TXN_FIELDS = ["amount_paise", "txn_date", "txn_id", "channel"]
NON_TXN_CATEGORIES = ["account_access", "kyc", "other"]

for p in [DATA / "train.jsonl", DATA / "dev.jsonl", DATA / "test_inputs.jsonl", PACK / "score.py", PACK / "SCHEMA.md"]:
    assert p.exists(), f"missing {p}"

print("root:", ROOT)
print("runs:", RUNS)

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def system_text(row):
    return row["messages"][0]["content"]


def user_text(row):
    return row["messages"][1]["content"]


def label_text(row):
    return row["messages"][2]["content"]


def ticket_body(row):
    return user_text(row).split("\n\n", 1)[-1]


def received_date(row):
    return datetime.strptime(user_text(row).split("\n", 1)[0].split()[1], "%Y-%m-%d").date()


def ticket_source(row):
    return user_text(row).split("\n")[1].split(": ", 1)[1].strip()


def canonical(label):
    return json.dumps({k: label[k] for k in KEYS}, ensure_ascii=False)


_spec = importlib.util.spec_from_file_location("nimbus_score", PACK / "score.py")
scorer = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(scorer)


def evaluate(gold_rows, pred_rows):
    return scorer.score(gold_rows, pred_rows)


def show_scores(res, prefixes=None):
    print(f"{'group':28} {'n':>5} {'valid':>7} {'exact':>7} {'fields':>7}")
    for name in sorted(res, key=lambda x: (x != "all", x)):
        if prefixes is not None and name != "all" and not name.startswith(tuple(prefixes)):
            continue
        r = res[name]
        print(f"{name:28} {r['n']:5d} {r['json_valid']:7.1%} {r['exact_match']:7.1%} {r['mean_field_acc']:7.1%}")
    print("per-field accuracy:", {k: round(v, 3) for k, v in res["all"]["fields"].items()})


EXPERIMENTS_CSV = RUNS / "experiments.csv"
EXPERIMENT_COLUMNS = ["run", "change", "train_rows", "dev_exact", "dev_field_acc", "dev_json_valid",
                      "peak_gpu_gb", "train_minutes", "trainable_params", "timestamp"]


def log_experiment(record):
    record = {**record, "timestamp": datetime.now().isoformat(timespec="seconds")}
    rows = []
    if EXPERIMENTS_CSV.exists():
        with open(EXPERIMENTS_CSV, newline="", encoding="utf-8") as f:
            rows = [r for r in csv.DictReader(f) if r["run"] != record["run"]]
    EXPERIMENTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    with open(EXPERIMENTS_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=EXPERIMENT_COLUMNS, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows + [record])
    return record


train_raw = load_jsonl(DATA / "train.jsonl")
dev = load_jsonl(DATA / "dev.jsonl")
test = load_jsonl(DATA / "test_inputs.jsonl")
print(len(train_raw), len(dev), len(test))
print(user_text(dev[0]))
print(label_text(dev[0]))

In [ ]:
import torch
import transformers


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)


set_seed()

print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    NATIVE_BF16 = props.major >= 8
    print("gpu:", props.name, f"{props.total_memory / 1e9:.1f} GB", f"| compute capability {props.major}.{props.minor}",
          "| native bf16:", NATIVE_BF16, "| training dtype:", "bf16" if NATIVE_BF16 else "fp16")
else:
    print("gpu: none")

## 1. Part A: data audit

The training labels came out of a noisy pipeline. The audit has three layers:

1. **Structure**: the label must parse as JSON, have the 8 spec keys, use the right types and formats, not repeat dev or test tickets, and not be a duplicate or a junk ticket.
2. **Rules**: a reference labeller written only from `SCHEMA.md` and the phrasing seen in `train.jsonl` reads each ticket and works out the fields the spec defines mechanically (priority, needs_human, txn_id, amount, date, channel). Every disagreement with the given label is a finding.
3. **Consistency**: tickets that open with the same issue sentence should share a category and a language. Rows that go against their group are flagged.

The reference labeller is only an audit tool. It is never used at inference. It was built from train only. Dev is used once at the end to measure how accurate the labeller is, not to tune it.

In [ ]:
import re
from collections import Counter, defaultdict

import pandas as pd

FIXED_SYSTEM = system_text(dev[0])
findings = defaultdict(set)


def flag(code, rid):
    findings[code].add(rid)


def parse_label(text):
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def norm_text(s):
    return re.sub(r"\s+", " ", s).strip().lower()


ISO_DATE = re.compile(r"\d{4}-\d{2}-\d{2}")
DMY_DATE = re.compile(r"(\d{2})/(\d{2})/(\d{4})")
TXN_ID = re.compile(r"NP\d{10}")

labels = {}
for r in train_raw:
    rid = r["id"]
    if [m["role"] for m in r["messages"]] != ["system", "user", "assistant"]:
        flag("bad_roles", rid)
    if system_text(r) != FIXED_SYSTEM:
        flag("bad_system_prompt", rid)
    y = parse_label(label_text(r))
    if y is None:
        flag("label_not_json", rid)
        continue
    labels[rid] = y
    if "prio" in y:
        flag("schema_v1_prio_key", rid)
    elif set(y) != set(KEYS):
        flag("wrong_keys", rid)
    if y.get("category") not in CATEGORIES:
        flag("bad_category_value", rid)
    if "priority" in y and y["priority"] not in PRIORITIES:
        flag("bad_priority_value", rid)
    a = y.get("amount_paise")
    if a is not None and type(a) is not int:
        flag("amount_wrong_type", rid)
    d = y.get("txn_date")
    if d is not None:
        if isinstance(d, str) and DMY_DATE.fullmatch(d):
            flag("date_dd_mm_yyyy", rid)
        elif not (isinstance(d, str) and ISO_DATE.fullmatch(d)):
            flag("date_bad_format", rid)
        elif d > str(received_date(r)):
            flag("date_after_received", rid)
    t = y.get("txn_id")
    if t is not None and not (isinstance(t, str) and TXN_ID.fullmatch(t)):
        flag("txn_id_not_normalized", rid)
    if y.get("channel") not in [None, *CHANNELS]:
        flag("bad_channel_value", rid)
    if y.get("language") not in LANGUAGES:
        flag("bad_language_value", rid)
    if type(y.get("needs_human")) is not bool:
        flag("needs_human_wrong_type", rid)
    if y.get("category") in NON_TXN_CATEGORIES and any(y.get(k) is not None for k in TXN_FIELDS):
        flag("txn_fields_on_non_txn_category", rid)

dev_users = {norm_text(user_text(r)) for r in dev}
test_users = {norm_text(user_text(r)) for r in test}
dev_bodies = {norm_text(ticket_body(r)) for r in dev}
test_bodies = {norm_text(ticket_body(r)) for r in test}
for r in train_raw:
    u, b = norm_text(user_text(r)), norm_text(ticket_body(r))
    if u in dev_users:
        flag("leak_dev_exact", r["id"])
    elif b in dev_bodies:
        flag("leak_dev_same_body", r["id"])
    if u in test_users:
        flag("leak_test_exact", r["id"])
    elif b in test_bodies:
        flag("leak_test_same_body", r["id"])

groups = defaultdict(list)
for r in train_raw:
    groups[norm_text(user_text(r))].append(r)
dup_groups = [g for g in groups.values() if len(g) > 1]
for g in dup_groups:
    distinct = {label_text(r) for r in g}
    for r in g[1:]:
        flag("duplicate_same_label" if len(distinct) == 1 else "duplicate_conflicting_label", r["id"])

JUNK = re.compile(r"(test|asdf|qwerty|hello|hi|\d|\W|_)*", re.I)
for r in train_raw:
    body = re.sub(r"^(\[IVR transcript\]|@NimbusPayCare)", "", ticket_body(r).strip()).strip()
    if len(body) < 20 and JUNK.fullmatch(body):
        flag("junk_ticket", r["id"])

print(f"train rows {len(train_raw)}, parsed labels {len(labels)}, duplicate groups {len(dup_groups)}")
structure_table = pd.DataFrame(
    [(k, len(v), ", ".join(sorted(v)[:4])) for k, v in sorted(findings.items(), key=lambda x: -len(x[1]))],
    columns=["finding", "rows", "examples"],
)
structure_table

In [ ]:
from datetime import date, timedelta

MONTHS = {m: i + 1 for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}
MON = r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*"

LEGAL = re.compile(r"\brbi\b|ombudsman|consumer court|lawyer|legal notice|legal action|\bvakil|\bcourt\b", re.I)
REPEAT = re.compile(
    r"third time|teesri baar|again and again|baar baar|contacted support about this before|"
    r"already written \d+ times|pehle bhi complaint|multiple times about this|second complaint|\d+(st|nd|rd|th)? (time|baar)\b",
    re.I,
)
HINGLISH = re.compile(
    r"\b(hai|hain|nahi|mera|mere|meri|maine|kya|kar|karo|karna|ho|hua|gaya|gaye|paise|paisa|tha|thi|kat|"
    r"wapas|abhi|baat|ki|ka|ke|ko|se|mein|bhi|kaise|kab|kyun|jaldi|raha|rahi|hoon|kal|parso|aaj|din|pehle|wala|"
    r"chahiye|lekin|liye|diya|dunga|jaunga|bol|baar|karun|karu|kitna|kyu|lagega|mila|mile|atka|hota|dhanyavaad)\b",
    re.I,
)
DISTRACTOR = re.compile(r"balance|limit|\bfee|charges? of|pade hain|abhi .* hain", re.I)
TXN_ID_LOOSE = re.compile(r"\bn\s*[\-.]?\s*p[\s\-_.:#]*((?:\d[\s\-]*){10})(?!\d)", re.I)
NUM = r"\d[\d,]*(?:\.\d+)?"
MONEY = re.compile(
    rf"(?:(?:rs\.?|inr|₹)\s*(?P<a>{NUM})\s*(?P<am>k\b|lakhs?\b)?(?:/-)?)"
    rf"|(?:(?P<b>{NUM})\s*(?P<bm>k|lakhs?)\b(?:\s*(?:rupees|rupaye|rs))?)"
    rf"|(?:(?P<c>{NUM})\s*(?:rupees|rupaye|rupaiye|rs\b))",
    re.I,
)
BARE_AMOUNT = re.compile(
    rf"(?:amount|rakam|total|value|it was for|it is)\D{{0,12}}?(?P<d>{NUM})(?![\d/\-])"
    rf"|(?P<e>{NUM})\s*(?:is the amount|was the value|in total|kat gaye|ka transaction|ka tha|tha\b)",
    re.I,
)
NOT_MONEY = re.compile(
    r"\bn\s*[\-.]?\s*p[\s\-_.:#]*(?:\d[\s\-]*){10}|#\d+|\b\d{9,}\b|\d+(?:\.\d+){2,}|\d{1,4}[/\-]\d{1,2}[/\-]\d{2,4}"
    r"|version \S+|(?:note|iphone|realme|pixel|vivo y|samsung m|samsung a|galaxy s|nord|oneplus)\s*\d+\w*"
    r"|\b\d{1,2}(st|nd|rd|th)?\s+" + MON + r"|" + MON + r"\s+\d{1,2}(st|nd|rd|th)?\b(\s+\d{4})?"
    r"|\d+\s*(days?|din|times|baar)\b|\b20\d\d\b",
    re.I,
)


def clean_body(row):
    body = ticket_body(row)
    lines = [ln for ln in body.split("\n") if not ln.lstrip().startswith(">") and not re.match(r"\s*On .* wrote:\s*$", ln)]
    lines = [ln for ln in lines if not ln.lower().startswith("subject:")]
    return "\n".join(lines)


def sentences(text):
    text = re.sub(r"\b(rs)\.\s*(?=\d)", r"\1 ", text, flags=re.I)
    text = re.sub(r"\s+(?=(fyi|my account balance|account mein abhi|ref:|id:|bank ref)\b)", "\n", text, flags=re.I)
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]


def to_paise(num, mult):
    value = float(num.replace(",", ""))
    mult = (mult or "").lower()
    if mult == "k":
        value *= 1_000
    elif mult.startswith("lakh"):
        value *= 100_000
    return int(round(value * 100))


def extract_amount(text):
    found = []
    for s in sentences(text):
        if DISTRACTOR.search(s):
            continue
        masked = NOT_MONEY.sub(lambda m: " " * len(m.group(0)), s)
        hits = [to_paise(m.group("a") or m.group("b") or m.group("c"), m.group("am") or m.group("bm")) for m in MONEY.finditer(masked)]
        if not hits:
            hits = [to_paise(m.group("d") or m.group("e"), None) for m in BARE_AMOUNT.finditer(masked)]
        found.extend(hits)
    found = list(dict.fromkeys(found))
    return found[0] if len(found) == 1 else (None if not found else "ambiguous")


def latest_on_or_before(day, month, ref):
    for year in (ref.year, ref.year - 1):
        try:
            d = date(year, month, day)
        except ValueError:
            continue
        if d <= ref:
            return d
    return None


def extract_date(text, ref):
    t = text.lower()
    hits = []
    for m in re.finditer(r"\b(\d{1,2})[/\-](\d{1,2})[/\-](\d{2,4})\b", t):
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        y = y + 2000 if y < 100 else y
        try:
            hits.append(date(y, mo, d))
        except ValueError:
            pass
    for m in re.finditer(r"\b(\d{1,2})(?:st|nd|rd|th)?\s+" + MON + r"\b(?:\s+(\d{4}))?", t):
        day, mon, yr = int(m.group(1)), MONTHS[m.group(2)], m.group(3)
        hits.append(date(int(yr), mon, day) if yr else latest_on_or_before(day, mon, ref))
    for m in re.finditer(r"\b" + MON + r"\s+(\d{1,2})(?:st|nd|rd|th)?\b(?:,?\s+(\d{4}))?", t):
        mon, day, yr = MONTHS[m.group(1)], int(m.group(2)), m.group(3)
        hits.append(date(int(yr), mon, day) if yr else latest_on_or_before(day, mon, ref))
    for m in re.finditer(r"\b(\d+)\s*(?:days? ago|din pehle)", t):
        hits.append(ref - timedelta(days=int(m.group(1))))
    if re.search(r"day before yesterday|\bparso\b", t):
        hits.append(ref - timedelta(days=2))
    elif re.search(r"\byesterday\b|\bkal\b", t):
        hits.append(ref - timedelta(days=1))
    if re.search(r"\btoday\b|\baaj\b", t):
        hits.append(ref)
    hits = list(dict.fromkeys(h for h in hits if h))
    return hits[0].isoformat() if len(hits) == 1 else (None if not hits else "ambiguous")


def extract_txn_id(text):
    ids = list(dict.fromkeys("NP" + re.sub(r"\D", "", m.group(1)) for m in TXN_ID_LOOSE.finditer(text)))
    return ids[0] if len(ids) == 1 else (None if not ids else "ambiguous")


def extract_channel(text):
    t = text.lower()
    found = set()
    if re.search(r"\bupi\b", t):
        found.add("upi")
    if re.search(r"net\s?banking|internet banking", t):
        found.add("netbanking")
    if re.search(r"\bwallet\b", t):
        found.add("wallet")
    t_card = re.sub(r"\bs\w*c\w*h card", " ", t)
    if re.search(r"\bcard\b|\bvisa\b|mastercard", t_card):
        found.add("card")
    return found.pop() if len(found) == 1 else (None if not found else "ambiguous")


def is_hinglish(text):
    body = re.sub(r"^(\[IVR transcript\]|@NimbusPayCare)\s*", "", text.strip())
    body = re.sub(r"^(hi|hello|hey|sir|bhai|namaste|hello ji|hi team|good morning|dear nimbuspay support|urgent!!)[,!]?\s*", "", body, flags=re.I)
    words = re.findall(r"[a-z]+", body.lower())
    hits = sum(bool(HINGLISH.fullmatch(w)) for w in words)
    return hits >= 2


def rule_priority(category, amount, text):
    amount = amount if type(amount) is int else 0
    if category == "fraud" or amount >= 5_000_000 or LEGAL.search(text):
        return "P1"
    if (category in ("payment_failed", "refund") and amount >= 500_000) or category == "account_access":
        return "P2"
    return "P3"


def rule_needs_human(priority, text):
    return priority == "P1" or bool(REPEAT.search(text))


def reference_fields(row, category):
    text = clean_body(row)
    ref = received_date(row)
    out = {
        "language": "hi-en" if is_hinglish(text) else "en",
        "legal": bool(LEGAL.search(text)),
        "repeat": bool(REPEAT.search(text)),
    }
    if category in NON_TXN_CATEGORIES:
        out.update(amount_paise=None, txn_date=None, txn_id=None, channel=None)
    else:
        out.update(amount_paise=extract_amount(text), txn_date=extract_date(text, ref),
                   txn_id=extract_txn_id(text), channel=extract_channel(text))
    return out


demo = train_raw[0]
print(user_text(demo))
print(reference_fields(demo, "payment_failed"))

In [ ]:
def iso_from_label_date(d):
    m = DMY_DATE.fullmatch(d) if isinstance(d, str) else None
    return f"{m.group(3)}-{m.group(2)}-{m.group(1)}" if m else d


def label_txn_id(t):
    return "NP" + re.sub(r"\D", "", t) if isinstance(t, str) else t


def is_unit_error(label_amount, rule_amount):
    return (type(label_amount) is int and type(rule_amount) is int and label_amount > 0
            and label_amount != rule_amount and rule_amount // 100 == label_amount)


usable = [r for r in train_raw if r["id"] in labels and r["id"] not in findings["junk_ticket"]]
reference = {}
disagree = defaultdict(list)
for r in usable:
    rid, y = r["id"], labels[r["id"]]
    text = clean_body(r)
    ref = reference_fields(r, y["category"])
    amount = ref["amount_paise"] if type(ref["amount_paise"]) is int else y.get("amount_paise")
    ref["priority"] = rule_priority(y["category"], amount, text)
    ref["needs_human"] = rule_needs_human(ref["priority"], text)
    reference[rid] = ref
    given = {**y, "txn_date": iso_from_label_date(y.get("txn_date")), "txn_id": label_txn_id(y.get("txn_id"))}
    for k in ["amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]:
        if given.get(k) != ref[k]:
            disagree[k].append(rid)
    if "priority" in y and y["priority"] != ref["priority"]:
        disagree["priority"].append(rid)

unit_errors = [rid for rid in disagree["amount_paise"] if is_unit_error(labels[rid].get("amount_paise"), reference[rid]["amount_paise"])]
print(f"rows compared: {len(usable)}")
print(pd.DataFrame(
    [(k, len(v), len(v) / len(usable)) for k, v in disagree.items()] + [("amount_paise: of which rupees-not-paise", len(unit_errors), len(unit_errors) / len(usable))],
    columns=["field", "label != rule", "rate"],
).to_string(index=False))

print("\nv1 'prio' value vs rule priority:")
v1_ids = sorted(findings["schema_v1_prio_key"])
print(pd.crosstab(pd.Series([labels[i]["prio"] for i in v1_ids], name="prio"),
                  pd.Series([reference[i]["priority"] for i in v1_ids], name="rule")))

print("\npriority given vs rule (non-v1 rows):")
pri_rows = [r["id"] for r in usable if "priority" in labels[r["id"]]]
print(pd.crosstab(pd.Series([labels[i]["priority"] for i in pri_rows], name="given"),
                  pd.Series([reference[i]["priority"] for i in pri_rows], name="rule")))

bad_pri = set(disagree["priority"])
nh_ok = sum(labels[i]["needs_human"] == reference[i]["needs_human"] for i in bad_pri)
print(f"\nrows whose priority breaks the rules: {len(bad_pri)}; needs_human agrees with the RULE priority on {nh_ok} of them")
print("so the corruption hit priority only, and needs_human still carries the original priority")

GREETING = re.compile(r"^((\[ivr transcript\]|@nimbuspaycare|urgent!!|hi team|hello ji|good morning|dear nimbuspay support|namaste|hello|hey|bhai|sir|hi)\s*[,!]?\s*)+", re.I)


def issue_sentence(row):
    s = sentences(GREETING.sub("", re.sub(r"\s+", " ", clean_body(row)).strip()))
    out = s[0] if s else ""
    if len(out.split()) < 4 and len(s) > 1:
        out += " " + s[1]
    return out.lower()


by_issue = defaultdict(list)
for r in usable:
    by_issue[issue_sentence(r)].append(r["id"])
multi = {k: v for k, v in by_issue.items() if len(v) > 1}
conflicts = {k: Counter(labels[i]["category"] for i in v) for k, v in multi.items() if len({labels[i]["category"] for i in v}) > 1}
print(f"\ncategory consistency: {len(multi)} issue sentences shared by 2+ rows, {len(conflicts)} with mixed categories")
for k, c in conflicts.items():
    print("  ", dict(c), "|", k[:100])

for k in ["priority", "amount_paise"]:
    print(f"\nexamples, {k}:")
    for rid in disagree[k][:5]:
        r = next(x for x in usable if x["id"] == rid)
        print(f"  {rid} given={labels[rid].get(k)} rule={reference[rid][k]} | {re.sub(chr(10), ' ', clean_body(r))[:160]}")

In [ ]:
DROP_ORDER = [
    ("leak_dev_exact", "same ticket as a dev row; training on it would leak dev"),
    ("leak_dev_same_body", "same ticket body as a dev row; training on it would leak dev"),
    ("leak_test_exact", "same ticket as a test row"),
    ("leak_test_same_body", "same ticket body as a test row"),
    ("junk_ticket", "placeholder text such as 'test' or 'asdf' with a made-up label"),
    ("label_not_json", "label is cut off and does not parse as JSON"),
    ("duplicate_conflicting_label", "exact duplicate ticket with a different label"),
    ("duplicate_same_label", "exact duplicate of an earlier row"),
]
V1_PRIORITY = {"high": "P1", "medium": "P2", "low": "P3"}

decisions = {}
fixed_labels = {}
for r in train_raw:
    rid = r["id"]
    drop = [code for code, _ in DROP_ORDER if rid in findings[code]]
    if drop:
        decisions[rid] = ("drop", drop[0])
        continue
    y = dict(labels[rid])
    reasons = []
    if "prio" in y:
        y["priority"] = V1_PRIORITY[y.pop("prio")]
        reasons.append("schema_v1_prio_key")
    if rid in findings["date_dd_mm_yyyy"]:
        y["txn_date"] = iso_from_label_date(y["txn_date"])
        reasons.append("date_dd_mm_yyyy")
    if rid in findings["txn_id_not_normalized"]:
        y["txn_id"] = label_txn_id(y["txn_id"])
        reasons.append("txn_id_not_normalized")
    if rid in unit_errors:
        y["amount_paise"] = reference[rid]["amount_paise"]
        reasons.append("amount_in_rupees_not_paise")
    if y["priority"] != reference[rid]["priority"]:
        y["priority"] = reference[rid]["priority"]
        reasons.append("priority_breaks_rules")
    if y["needs_human"] != reference[rid]["needs_human"]:
        y["needs_human"] = reference[rid]["needs_human"]
        reasons.append("needs_human_breaks_rules")
    fixed_labels[rid] = {k: y[k] for k in KEYS}
    if reasons:
        decisions[rid] = ("fix", ";".join(reasons))

REASON_TEXT = dict(DROP_ORDER) | {
    "schema_v1_prio_key": "old v1 schema: key 'prio' with low/medium/high; renamed to 'priority' and mapped to P3/P2/P1",
    "date_dd_mm_yyyy": "txn_date written DD/MM/YYYY; converted to YYYY-MM-DD",
    "txn_id_not_normalized": "txn_id had lowercase, spaces or hyphens; normalized to NP + 10 digits",
    "amount_in_rupees_not_paise": "amount stored in rupees (decimals cut off); replaced with the paise amount stated in the ticket",
    "priority_breaks_rules": "priority contradicts the spec given category, amount and legal threat; recomputed",
    "needs_human_breaks_rules": "needs_human contradicts the spec; recomputed",
}

log = pd.DataFrame([(rid, a, reason) for rid, (a, reason) in sorted(decisions.items())], columns=["id", "action", "reason"])
print(log["action"].value_counts().to_string())
reason_counts = Counter(code for reason in log["reason"] for code in reason.split(";"))
print(pd.DataFrame([(c, n, REASON_TEXT[c]) for c, n in reason_counts.most_common()], columns=["reason", "rows", "meaning"]).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

dev_ids_leaked = findings["leak_dev_exact"] | findings["leak_dev_same_body"] | findings["leak_test_exact"] | findings["leak_test_same_body"]
train_clean = [
    {"id": r["id"], "messages": [r["messages"][0], r["messages"][1], {"role": "assistant", "content": canonical(fixed_labels[r["id"]])}]}
    for r in train_raw if r["id"] in fixed_labels
]
train_noleak = [r for r in train_raw if r["id"] not in dev_ids_leaked]

CLEAN_DIR = RUNS / "data"
save_jsonl(train_clean, CLEAN_DIR / "train_clean.jsonl")
save_jsonl(train_noleak, CLEAN_DIR / "train_raw_noleak.jsonl")
log.to_csv(ROOT / "cleaning_log.csv", index=False)


def violations(rows):
    out = Counter()
    for r in rows:
        y = parse_label(label_text(r))
        if y is None or list(y) != KEYS:
            out["structure"] += 1
            continue
        if label_text(r) != canonical(y):
            out["format"] += 1
        if not (y["txn_id"] is None or TXN_ID.fullmatch(y["txn_id"])):
            out["txn_id_format"] += 1
        if not (y["txn_date"] is None or ISO_DATE.fullmatch(y["txn_date"])):
            out["date_format"] += 1
        ref = reference_fields(r, y["category"])
        amount = ref["amount_paise"] if type(ref["amount_paise"]) is int else y["amount_paise"]
        pri = rule_priority(y["category"], amount, clean_body(r))
        for k in ["amount_paise", "txn_date", "txn_id", "channel", "language"]:
            if ref[k] != y[k]:
                out[k] += 1
        out["priority"] += pri != y["priority"]
        out["needs_human"] += rule_needs_human(pri, clean_body(r)) != y["needs_human"]
    return out


before, after = violations(train_noleak), violations(train_clean)
summary = pd.DataFrame({"raw (dev leaks removed)": before, "clean": after}).fillna(0).astype(int)
summary.loc["rows"] = [len(train_noleak), len(train_clean)]
print(summary.to_string())

dev_check = violations(dev)
dev_bad = {k: v for k, v in dev_check.items() if v}
print(f"\nreference labeller vs clean dev labels ({len(dev)} rows), disagreements per field: {dev_bad or 'none'}")

users = [norm_text(user_text(r)) for r in train_clean]
assert len(users) == len(set(users)), "duplicates left"
assert not (set(users) & dev_users), "dev leak left"
assert sum(after.values()) == 0, "clean set still breaks a rule"

counts = Counter(code for reason in log["reason"] for code in reason.split(";"))
codes = [c for c, _ in counts.most_common()][::-1]
colors = ["#c0392b" if any(c == d for d, _ in DROP_ORDER) else "#2471a3" for c in codes]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(codes, [counts[c] for c in codes], color=colors)
for i, c in enumerate(codes):
    ax.text(counts[c] + 1, i, str(counts[c]), va="center", fontsize=9)
ax.set_xlabel("training rows")
ax.set_title(f"Where the noise was: {sum(log.action == 'drop')} rows dropped (red), {sum(log.action == 'fix')} rows fixed (blue)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RUNS / "audit_noise.png", dpi=150)
plt.show()

print(f"train_clean: {len(train_clean)} rows -> {CLEAN_DIR / 'train_clean.jsonl'}")
print(f"train_raw_noleak: {len(train_noleak)} rows (only dev leaks removed, for the raw-vs-clean experiment)")
print(f"cleaning_log.csv: {len(log)} rows -> {ROOT / 'cleaning_log.csv'}")

## 2. Chat template and sequence length

At test time the model gets the `system` and `user` messages formatted with Qwen's own chat template, followed by the assistant header. It must then write the JSON and stop. Training uses exactly the same formatting:

`prompt = apply_chat_template([system, user], add_generation_prompt=True)`, then `target = label_json + "<|im_end|>"`

Loss is computed only on `target`. `<|im_end|>` is included so the model learns to stop, because it is the tokenizer's EOS token and `generate` halts on it.

The training sequence length is chosen from the measured token lengths below, not guessed. Long tickets carry quoted email threads, and the test set has them too, so they cannot be cut.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
END = "<|im_end|>"
END_ID = tokenizer.convert_tokens_to_ids(END)
assert tokenizer.eos_token == END


def prompt_messages(row, system=None):
    return [{"role": "system", "content": system or system_text(row)}, {"role": "user", "content": user_text(row)}]


def prompt_string(row, system=None):
    return tokenizer.apply_chat_template(prompt_messages(row, system), tokenize=False, add_generation_prompt=True)


def encode_example(row):
    prompt_ids = tokenizer(prompt_string(row), add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(label_text(row) + END, add_special_tokens=False)["input_ids"]
    return {
        "input_ids": prompt_ids + target_ids,
        "labels": [-100] * len(prompt_ids) + target_ids,
        "prompt_len": len(prompt_ids),
        "target_len": len(target_ids),
    }


example = train_clean[0]
print(prompt_string(example) + label_text(example) + END)

mismatch = 0
for r in train_clean:
    full = tokenizer.apply_chat_template(r["messages"], tokenize=False)
    full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    enc = encode_example(r)
    if full_ids[: len(enc["input_ids"])] != enc["input_ids"]:
        mismatch += 1
print(f"\nprompt + target tokenization identical to the full chat template on {len(train_clean) - mismatch}/{len(train_clean)} rows")
assert mismatch == 0

enc = encode_example(example)
trained_on = tokenizer.decode([t for t, l in zip(enc["input_ids"], enc["labels"]) if l != -100])
print("tokens with loss:", repr(trained_on))
assert trained_on == label_text(example) + END

In [ ]:
def prompt_len(row):
    return len(tokenizer(prompt_string(row), add_special_tokens=False)["input_ids"])


lengths = {
    "train prompt": [prompt_len(r) for r in train_clean],
    "dev prompt": [prompt_len(r) for r in dev],
    "test prompt": [prompt_len(r) for r in test],
    "train answer": [encode_example(r)["target_len"] for r in train_clean],
    "dev answer": [len(tokenizer(label_text(r) + END, add_special_tokens=False)["input_ids"]) for r in dev],
}
pcts = [50, 90, 95, 99, 100]
table = pd.DataFrame({name: np.percentile(v, pcts).round().astype(int) for name, v in lengths.items()}, index=[f"p{p}" for p in pcts]).T
table["mean"] = [round(float(np.mean(v)), 1) for v in lengths.values()]
print(table.to_string())

train_total = [p + a for p, a in zip(lengths["train prompt"], lengths["train answer"])]
quoted = sum(any(ln.lstrip().startswith(">") for ln in user_text(r).split("\n")) for r in train_clean)
long_rows = sum(t > 512 for t in train_total)
print(f"\ntrain rows with a quoted email: {quoted}; train sequences over 512 tokens: {long_rows} ({long_rows / len(train_total):.1%})")
print(f"test prompts over 512 tokens: {sum(t > 512 for t in lengths['test prompt'])}")

MAX_LEN = int(np.ceil(max(train_total) / 64) * 64)
assert max(lengths["train answer"] + lengths["dev answer"]) < MAX_NEW_TOKENS
print(f"\nlongest train sequence {max(train_total)} tokens -> MAX_LEN = {MAX_LEN} (no row is truncated)")
print(f"longest answer {max(lengths['train answer'] + lengths['dev answer'])} tokens, well inside max_new_tokens={MAX_NEW_TOKENS}")
print(f"longest test prompt {max(lengths['test prompt'])} tokens; inference never truncates, so this only needs to fit in memory")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
bins = np.logspace(np.log10(40), np.log10(max(lengths["test prompt"] + train_total) * 1.1), 40)
for name, color in [("train prompt", "#2471a3"), ("dev prompt", "#27ae60"), ("test prompt", "#c0392b")]:
    axes[0].hist(lengths[name], bins=bins, alpha=0.55, label=name, color=color)
axes[0].set_xscale("log")
axes[0].axvline(MAX_LEN, color="black", ls="--", lw=1)
axes[0].text(MAX_LEN, axes[0].get_ylim()[1] * 0.9, f" MAX_LEN={MAX_LEN}", fontsize=8)
axes[0].set_xlabel("prompt tokens (log scale)")
axes[0].set_ylabel("tickets")
axes[0].legend(fontsize=8)
axes[0].set_title("Prompt length: short tickets plus a long quoted-email tail")
axes[1].hist(lengths["train answer"], bins=30, color="#7d3c98")
axes[1].set_xlabel("answer tokens (JSON + <|im_end|>)")
axes[1].set_title("Answer length")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RUNS / "token_lengths.png", dpi=150)
plt.show()

## 3. Part B: baselines

Two reference points on dev, with the untouched base model in fp16:

1. **Fixed prompt**: exactly what the fine-tuned model will see (the one-line system prompt and the ticket).
2. **Rules in the prompt**: the full `SCHEMA.md` appended to the system prompt. This is allowed only here, as a reference for how far prompting alone gets.

Inference follows the evaluation protocol: Qwen's chat template, `model.generate` with greedy decoding (`do_sample=False`) and `max_new_tokens=200`, and the output only stripped of whitespace. Everything else stays at the model's own `generation_config`. That includes Qwen's default `repetition_penalty=1.05`, which still applies under greedy decoding. This is what plain `model.generate` does, so it is what the evaluation will do.

Batches are formed from tickets of similar length and left-padded, to keep generation fast on a T4. Step 7 checks that batched outputs equal one-at-a-time outputs.

In [ ]:
import gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GEN_KWARGS = {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS}


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def load_model(model_id=BASE_MODEL, four_bit=False, adapter=None):
    kwargs = {"dtype": DTYPE}
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
        kwargs["device_map"] = {"": 0}
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if not four_bit:
        model.to(DEVICE)
    if adapter is not None:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(adapter))
    model.eval()
    return model


def length_batches(encoded, token_budget, max_batch):
    order = sorted(range(len(encoded)), key=lambda i: len(encoded[i]))
    batches, current = [], []
    for i in order:
        longest = max([len(encoded[j]) for j in current] + [len(encoded[i])])
        if current and ((longest + MAX_NEW_TOKENS) * (len(current) + 1) > token_budget or len(current) == max_batch):
            batches.append(current)
            current = []
        current.append(i)
    if current:
        batches.append(current)
    return batches


@torch.no_grad()
def generate_predictions(model, rows, system=None, token_budget=24_000, max_batch=32, extra_gen=None, desc="generate"):
    tokenizer.padding_side = "left"
    prompts = [prompt_string(r, system) for r in rows]
    encoded = [tokenizer(p, add_special_tokens=False)["input_ids"] for p in prompts]
    outputs = [None] * len(rows)
    batches = length_batches(encoded, token_budget, max_batch)
    start = time.time()
    for b, idx in enumerate(batches):
        batch = tokenizer.pad({"input_ids": [encoded[i] for i in idx]}, return_tensors="pt").to(model.device)
        gen = model.generate(**batch, **GEN_KWARGS, **(extra_gen or {}), pad_token_id=tokenizer.pad_token_id)
        new_tokens = gen[:, batch["input_ids"].shape[1]:]
        for i, toks in zip(idx, new_tokens):
            outputs[i] = tokenizer.decode(toks, skip_special_tokens=True).strip()
        print(f"\r{desc}: batch {b + 1}/{len(batches)} ({time.time() - start:.0f}s)", end="")
    print()
    return [{"id": r["id"], "output": o} for r, o in zip(rows, outputs)]


def effective_generation_config(model):
    cfg = model.generation_config.to_dict()
    cfg.update(GEN_KWARGS)
    keep = ["do_sample", "max_new_tokens", "repetition_penalty", "temperature", "top_p", "top_k", "eos_token_id"]
    return {k: cfg.get(k) for k in keep}


def run_eval(name, model, rows=None, system=None, change="", extra=None, force=False, **gen):
    rows = rows if rows is not None else dev
    out_dir = RUNS / name
    pred_path = out_dir / "dev_predictions.jsonl"
    if pred_path.exists() and not force:
        preds = load_jsonl(pred_path)
        print(f"{name}: reusing saved predictions from {pred_path}")
    else:
        preds = generate_predictions(model, rows, system=system, desc=name, **gen)
        save_jsonl(preds, pred_path)
    res = evaluate(rows, preds)
    (out_dir / "dev_scores.json").write_text(json.dumps(res, indent=1), encoding="utf-8")
    show_scores(res, prefixes=["category="])
    log_experiment({
        "run": name, "change": change,
        "dev_exact": round(res["all"]["exact_match"], 4),
        "dev_field_acc": round(res["all"]["mean_field_acc"], 4),
        "dev_json_valid": round(res["all"]["json_valid"], 4),
        **(extra or {}),
    })
    return preds, res

In [ ]:
SCHEMA_TEXT = (PACK / "SCHEMA.md").read_text(encoding="utf-8")
RULES_SYSTEM = FIXED_SYSTEM + "\n\nFollow this labelling spec exactly and output only the JSON object.\n\n" + SCHEMA_TEXT
print(f"rules prompt: {len(tokenizer(RULES_SYSTEM)['input_ids'])} tokens added to every request")

free_memory()
base = load_model()
print("effective generation settings:", effective_generation_config(base))

print("\nBaseline 1: fixed prompt")
b1_preds, b1 = run_eval("B1_base_fixed_prompt", base, change="base model, fixed prompt, no fine-tuning")

print("\nBaseline 2: rules in the prompt")
b2_preds, b2 = run_eval("B2_base_rules_in_prompt", base, system=RULES_SYSTEM, change="base model, SCHEMA.md in system prompt",
                        token_budget=16_000, max_batch=8)

baseline_table = pd.DataFrame({
    "B1 fixed prompt": {"json_valid": b1["all"]["json_valid"], "exact_match": b1["all"]["exact_match"],
                        "mean_field_acc": b1["all"]["mean_field_acc"], **b1["all"]["fields"]},
    "B2 rules in prompt": {"json_valid": b2["all"]["json_valid"], "exact_match": b2["all"]["exact_match"],
                           "mean_field_acc": b2["all"]["mean_field_acc"], **b2["all"]["fields"]},
}).round(3)
print(baseline_table.to_string())

for name, preds in [("B1", b1_preds), ("B2", b2_preds)]:
    print(f"\n{name} sample outputs:")
    for p, g in list(zip(preds, dev))[:3]:
        print(f"  {p['id']} gold: {label_text(g)}")
        print(f"  {' ' * len(p['id'])} pred: {p['output'][:300]!r}")

del base
free_memory()

## 4. Part C: fine-tuning pipeline

A short, hand-written training loop rather than a framework trainer, so every choice is visible and easy to defend:

| Choice | Value | Why |
|---|---|---|
| Base | `Qwen2.5-3B-Instruct` in fp16 | the T4 has no native bf16, and the 3B model fits in fp16 with room to spare |
| Adapter | LoRA r=16, alpha=32, dropout 0.05 on all attention and MLP projections | ≈30M trainable params, under the 50M cap (r=32 would exceed it) |
| Loss | cross-entropy on the answer JSON and `<|im_end|>` only | the model learns to answer, not to repeat tickets, and learns to stop |
| Batches | 16 tickets per optimizer step, grouped by length, split into micro-batches of at most 8,192 tokens | short tickets do not pay for the padding of quoted-email tickets, and one long row never runs out of memory |
| Logits | the LM head runs only on answer positions | a 152k-word vocabulary over every prompt token would be the largest tensor in training; this skips it |
| Optimizer | AdamW, LR 2e-4, cosine schedule, 3% warmup, no weight decay, gradient clip 1.0, 3 epochs, seed 42 | standard LoRA settings; Part D tests rank and epochs |
| Precision | fp16 autocast with a gradient scaler; LoRA weights kept in fp32 | a stable fp16 recipe for pre-Ampere GPUs |
| Memory | gradient checkpointing | room for the 2,240-token rows |

Each optimizer step averages the loss over every answer token in its 16 tickets, however they are split into micro-batches, so the update does not depend on the batching. Dev loss is measured after each epoch to monitor fitting, never to train.

After training, the adapter is saved in fp16, under 100 MB, the same way it ships in `adapter/`. It is then **loaded fresh onto a clean base model** and scored on dev with the same protocol as the baselines, as a reviewer would do. Every run writes `train_log.csv`, `train_meta.json`, its adapter, its dev predictions and scores under `runs/<name>/`, and appends one row to `runs/experiments.csv`. A run whose adapter is already on Drive is reused, so a Colab disconnect costs nothing.

In [ ]:
import contextlib
import math
import torch.nn.functional as F
import matplotlib.pyplot as plt
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import get_cosine_schedule_with_warmup

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
TRAINABLE_CAP = 50_000_000


def collate(examples):
    width = max(len(e["input_ids"]) for e in examples)
    ids = torch.full((len(examples), width), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((len(examples), width), -100, dtype=torch.long)
    mask = torch.zeros((len(examples), width), dtype=torch.long)
    for k, e in enumerate(examples):
        n = len(e["input_ids"])
        ids[k, :n] = torch.tensor(e["input_ids"])
        labels[k, :n] = torch.tensor(e["labels"])
        mask[k, :n] = 1
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}


def plan_steps(lengths, rows_per_step, token_budget, rng):
    order = sorted(range(len(lengths)), key=lambda i: (lengths[i] // 32, rng.random()))
    chunks = [order[i:i + rows_per_step] for i in range(0, len(order), rows_per_step)]
    rng.shuffle(chunks)
    steps = []
    for chunk in chunks:
        micro, current = [], []
        for i in sorted(chunk, key=lambda i: lengths[i]):
            if current and lengths[i] * (len(current) + 1) > token_budget:
                micro.append(current)
                current = []
            current.append(i)
        micro.append(current)
        steps.append(micro)
    return steps


def autocast():
    if torch.cuda.is_available() and DTYPE != torch.float32:
        return torch.autocast("cuda", dtype=DTYPE)
    return contextlib.nullcontext()


def answer_loss(model, batch):
    inner = model.get_base_model()
    hidden = inner.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], use_cache=False).last_hidden_state
    targets = batch["labels"][:, 1:]
    keep = targets != -100
    logits = inner.lm_head(hidden[:, :-1][keep]).float()
    return F.cross_entropy(logits, targets[keep], reduction="sum"), int(keep.sum())


@torch.no_grad()
def dev_loss(model, encoded, token_budget):
    model.eval()
    lengths = [len(e["input_ids"]) for e in encoded]
    total, count = 0.0, 0
    for micro in plan_steps(lengths, len(encoded), token_budget, random.Random(0)):
        for mb in micro:
            batch = {k: v.to(model.device) for k, v in collate([encoded[i] for i in mb]).items()}
            with autocast():
                loss_sum, n = answer_loss(model, batch)
            total += loss_sum.item()
            count += n
    model.train()
    return total / count


def train_lora(name, rows, rank=16, epochs=3, lr=2e-4, four_bit=False, rows_per_step=16, token_budget=8192,
               warmup_ratio=0.03, seed=SEED, model_id=None, force=False):
    out_dir = RUNS / name
    adapter_dir = out_dir / "adapter"
    meta_path = out_dir / "train_meta.json"
    if meta_path.exists() and (adapter_dir / "adapter_config.json").exists() and not force:
        print(f"{name}: reusing trained adapter from {adapter_dir}")
        return json.loads(meta_path.read_text(encoding="utf-8"))

    set_seed(seed)
    free_memory()
    model = load_model(model_id or BASE_MODEL, four_bit=four_bit)
    model.config.use_cache = False
    checkpointing = {"use_reentrant": False}
    if four_bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs=checkpointing)
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=checkpointing)
    config = LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.05, target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, config)
    params = [p for p in model.parameters() if p.requires_grad]
    trainable = sum(p.numel() for p in params)
    assert trainable <= TRAINABLE_CAP, trainable
    assert all(p.dtype == torch.float32 for p in params)
    print(f"{name}: {len(rows)} rows, {'4-bit NF4' if four_bit else str(DTYPE).split('.')[-1]} base, LoRA r={rank}, trainable params {trainable:,}")

    encoded = [encode_example(r) for r in rows]
    lengths = [len(e["input_ids"]) for e in encoded]
    dev_encoded = [encode_example(r) for r in dev]
    total_steps = math.ceil(len(rows) / rows_per_step) * epochs
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(optimizer, math.ceil(warmup_ratio * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available() and DTYPE == torch.float16)
    rng = random.Random(seed)
    history, step = [], 0
    start = time.time()
    model.train()
    for epoch in range(1, epochs + 1):
        for micro in plan_steps(lengths, rows_per_step, token_budget, rng):
            n_target = sum(encoded[i]["target_len"] for mb in micro for i in mb)
            step_loss = 0.0
            for mb in micro:
                batch = {k: v.to(model.device) for k, v in collate([encoded[i] for i in mb]).items()}
                with autocast():
                    loss_sum, _ = answer_loss(model, batch)
                scaler.scale(loss_sum / n_target).backward()
                step_loss += loss_sum.item()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            step += 1
            history.append({"step": step, "epoch": epoch, "loss": step_loss / n_target,
                            "lr": scheduler.get_last_lr()[0], "grad_norm": grad_norm, "dev_loss": None})
            print(f"\r{name}: epoch {epoch}/{epochs} step {step}/{total_steps} loss {step_loss / n_target:.4f} "
                  f"({(time.time() - start) / 60:.1f} min)", end="")
        history[-1]["dev_loss"] = dev_loss(model, dev_encoded, token_budget)
        print(f"\n{name}: epoch {epoch} done, dev loss {history[-1]['dev_loss']:.4f}")

    minutes = (time.time() - start) / 60
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    for p in params:
        p.data = p.data.to(torch.float16)
    model.save_pretrained(adapter_dir)
    size_mb = sum(f.stat().st_size for f in adapter_dir.iterdir()) / 1e6
    meta = {"run": name, "train_rows": len(rows), "trainable_params": trainable, "peak_gpu_gb": round(peak, 2),
            "train_minutes": round(minutes, 1), "rank": rank, "epochs": epochs, "lr": lr, "four_bit": four_bit,
            "rows_per_step": rows_per_step, "steps": total_steps, "adapter_mb": round(size_mb, 1)}
    pd.DataFrame(history).to_csv(out_dir / "train_log.csv", index=False)
    meta_path.write_text(json.dumps(meta, indent=1), encoding="utf-8")
    print(f"{name}: trained in {minutes:.1f} min, peak GPU {peak:.1f} GB, adapter {size_mb:.0f} MB")
    del model, optimizer, scheduler, scaler, params
    free_memory()
    return meta


def train_and_eval(name, rows, change, four_bit=False, model_id=None, force=False, **train_kwargs):
    meta = train_lora(name, rows, four_bit=four_bit, model_id=model_id, force=force, **train_kwargs)
    model = load_model(model_id or BASE_MODEL, four_bit=four_bit, adapter=RUNS / name / "adapter")
    extra = {k: meta[k] for k in ["train_rows", "peak_gpu_gb", "train_minutes", "trainable_params"]}
    preds, res = run_eval(name, model, change=change, extra=extra, force=force)
    del model
    free_memory()
    return meta, preds, res


def plot_training(names, path):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for name in names:
        log = pd.read_csv(RUNS / name / "train_log.csv")
        axes[0].plot(log["step"], log["loss"].rolling(5, min_periods=1).mean(), label=name)
        ends = log.dropna(subset=["dev_loss"])
        axes[1].plot(ends["epoch"], ends["dev_loss"], marker="o", label=name)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("optimizer step")
    axes[0].set_ylabel("train loss (5-step mean)")
    axes[0].set_title("Training loss on answer tokens")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("dev loss")
    axes[1].set_title("Dev loss after each epoch")
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


def experiments_table():
    return pd.read_csv(EXPERIMENTS_CSV).drop(columns=["timestamp"]).set_index("run")

In [ ]:
train_lengths = [len(encode_example(r)["input_ids"]) for r in train_clean]
train_targets = [encode_example(r)["target_len"] for r in train_clean]
plan = plan_steps(train_lengths, 16, 8192, random.Random(SEED))
padded = sum(len(mb) * max(train_lengths[i] for i in mb) for micro in plan for mb in micro)
shuffled = list(range(len(train_clean)))
random.Random(SEED).shuffle(shuffled)
naive = sum(16 * max(train_lengths[i] for i in shuffled[k:k + 16]) for k in range(0, len(shuffled), 16))
print(f"optimizer steps per epoch: {len(plan)}, forward passes per epoch: {sum(len(m) for m in plan)}, "
      f"largest micro-batch: {max(len(mb) for m in plan for mb in m)} rows")
print(f"padding efficiency: {sum(train_lengths) / padded:.1%} with length grouping vs {sum(train_lengths) / naive:.1%} with random batches")
print(f"LM head runs on {sum(train_targets) / sum(train_lengths):.1%} of token positions (answer tokens only)")

REFERENCE = {"rank": 16, "epochs": 3, "lr": 2e-4, "rows_per_step": 16}
e2_meta, e2_preds, e2 = train_and_eval("E2_clean_lora_r16", train_clean, change="cleaned train, fp16 base, LoRA r16 (reference)", **REFERENCE)
plot_training(["E2_clean_lora_r16"], RUNS / "training_curves_reference.png")
print(experiments_table().to_string())